# Daily Rank-Average Ensemble (LB 0.94644)

This notebook is updated most days of the competition. The method never changes: take genuinely independent public submissions/blends, convert each to a 0-1 percentile rank, and take a weighted average. What changes day to day is which sources go in and at what weight, based on two checks. First: does a new top-scoring public notebook actually add diversity? We measure the Spearman rank-correlation between its predictions and our current best blend. Above ~0.999 it is almost certainly remixing the same underlying models (adding it will not move the score); notably lower (we look for <=0.998) means it was built with a genuinely different feature/model pipeline and is worth blending in. Second: if it passes that check, sweep the mixing weight on the public leaderboard rather than guessing a weight. ROC-AUC only depends on rank order, so ~1.0-correlated sources cannot move the score no matter how they are weighted; genuinely diverse sources show a real, LB-measurable optimum.

## Current recipe (2026-09-09)

Step one: base equals a 50/50 rank-average of nina2025's h-blend (https://www.kaggle.com/code/nina2025/ps-s6e9-h-blend-1, LB 0.94634) and jazivxt's zoom-zoom baseline (https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline, LB 0.94635), two independently-built blends of many base models each.

Step two: E equals 0.90 times base plus 0.05 times kospintr's stacked LGBM/CatBoost/XGB/HGBC (https://www.kaggle.com/code/kospintr/evehicle-stacked-lgbm-catb-xgb-hgbc-baseline) plus 0.05 times mikhailnaumov's XGBoost (https://www.kaggle.com/code/mikhailnaumov/electric-vehicle-purchases-xgb). This combination scored 0.94635 on the public LB.

Step three: F equals 0.50 times E plus 0.50 times najiama's Pure LGBM (https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637, LB 0.94638), a single LightGBM model built from scratch with a different feature-engineering pipeline (digit extraction, synthetic-artifact flags, and Markus's multi-scale Smooth Keys income/commute binning with triple target encoding). Its rank-correlation with E was only ~0.998 (vs. ~0.9995-0.99997 for every other public source we tried), which is why it was worth testing, and a weight sweep from 15% to 60% confirmed a real, LB-verified plateau at 0.94643 (50% and 60% scored identically).

Step four: final equals 0.90 times F plus 0.10 times yekenot's RealMLP - PyTorch (https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch), a single neural-network model (RealMLP architecture, not a gradient-boosted tree) with local OOF AUC 0.9459. It was the first genuinely different architecture family we tried (every other source, including najiama's, is still a tree ensemble). Its rank-correlation with F was only ~0.997, the lowest of any source so far, and a weight sweep from 10% to 20% found a plateau at 0.94644 (10% and 15% scored identically; 20% was already past the peak, back down to 0.94643).

## Credits

None of the blended submissions were produced in this notebook. All modeling credit goes to their authors: nina2025 (PS-s6e9 h-blend (1), https://www.kaggle.com/code/nina2025/ps-s6e9-h-blend-1); jazivxt (S6E9 Zoom Zoom Baseline / Single Model - Zoom Zoom, https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline); kospintr (EVehicle Stacked LGBM/CatB/XGB/HGBC, https://www.kaggle.com/code/kospintr/evehicle-stacked-lgbm-catb-xgb-hgbc-baseline); mikhailnaumov (Electric Vehicle Purchases XGB, https://www.kaggle.com/code/mikhailnaumov/electric-vehicle-purchases-xgb); najiama (Pure LGBM Model CV 0.94606 LB 0.94637, https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637); yekenot (PS|S6|E9: RealMLP - PyTorch, https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch).

To reproduce: add ps-s6e9-h-blend-1, evehicle-stacked-lgbm-catb-xgb-hgbc-baseline, electric-vehicle-purchases-xgb, pure-lgbm-model-cv-0-94606-lb-0-94637 and ps-s6-e9-realmlp-pytorch as Notebook inputs, and s6e9-zoom-zoom-baseline as a Dataset input (right sidebar, Add Input). Then run all cells.

In [ ]:
import pandas as pd
from scipy.stats import rankdata

## Load the source blends

In [ ]:
import glob
find_one = lambda pattern: glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)[0]
ID, TARGET = "id", "Will_Buy_EV"
nina = pd.read_csv(find_one("ps-s6e9-h-blend-1/submission.csv")).set_index(ID)[TARGET]
zoomzoom = pd.read_csv(find_one("submission_curvature9_lgbpair05.csv")).set_index(ID)[TARGET]
kospintr = pd.read_csv(find_one("evehicle-stacked-lgbm-catb-xgb-hgbc-baseline/submission.csv")).set_index(ID)[TARGET].reindex(nina.index)
mikhail = pd.read_csv(find_one("electric-vehicle-purchases-xgb/submission.csv")).set_index(ID)[TARGET].reindex(nina.index)
najiama = pd.read_csv(find_one("pure-lgbm-model-cv-0-94606-lb-0-94637/submission_LIGHTGBM.csv")).set_index(ID)[TARGET].reindex(nina.index)
realmlp = pd.read_csv(find_one("ps-s6-e9-realmlp-pytorch/submission.csv")).set_index(ID)[TARGET].reindex(nina.index)
nina.head()

# Weighted rank average

In [ ]:
rank01 = lambda s: pd.Series(rankdata(s) / len(s), index=s.index)
base = 0.5 * rank01(nina) + 0.5 * rank01(zoomzoom)
E = 0.90 * base + 0.05 * rank01(kospintr) + 0.05 * rank01(mikhail)
F = 0.50 * rank01(E) + 0.50 * rank01(najiama)
final = 0.90 * rank01(F) + 0.10 * rank01(realmlp)
final.head()

# Save submission

In [ ]:
submission = pd.DataFrame({ID: final.index, TARGET: final.to_numpy()})
submission.to_csv("submission.csv", index=False)
submission.head()